# Companion: OpenMM Production and MuPT Round-Trip

This notebook consumes the outputs from `DPD_Role_Aware_SAAMR_All_Atom_Ionomer_OpenMM.ipynb`. It treats the MuPT-generated SDF as the canonical molecular/hierarchy cache, uses the dense `production_start` files when available, runs OpenMM production MD, writes restart/analysis files, exports a final SDF/mmCIF snapshot, and reconstructs the MuPT universe from the final SDF.

Expected input layout for one system:

```text
examples_system/dpd_role_aware_ionomer_outputs/<system>/sdf/<system>.sdf
examples_system/dpd_role_aware_ionomer_outputs/<system>/OpenMM/production_start/*_manifest.json
examples_system/dpd_role_aware_ionomer_outputs/<system>/OpenMM/production_start/*_arrays.npz
```

If `production_start` is missing, rerun the builder notebook with `RUN_PERIODIC_NPT = True`. The SDF alone is enough to reconstruct MuPT/OpenFF molecules, but dense production MD should start from the equilibrated periodic coordinates and box.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import logging
import sys
import time

import numpy as np
from anytree import PreOrderIter
from rdkit import Chem
from rdkit.Geometry import Point3D


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

from mupt.interfaces.rdkit import primitive_from_mupt_sdf
from mupt.roles import PrimitiveRole

logging.getLogger("mupt.mupr.primitives").setLevel(logging.ERROR)

OUTPUT_ROOT = EXAMPLES_ROOT / "examples_system" / "dpd_role_aware_ionomer_outputs"
print(f"Repository root: {EXAMPLES_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

## 1. Production Knobs

Use short durations first. Once the handoff is validated, increase `PRODUCTION_DURATION_NS` and lower the reporting cadence to fit your storage budget.

In [ ]:
SYSTEM_NAME = "m5"
FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"

PRODUCTION_TEMPERATURE_K = 373.0
PRODUCTION_PRESSURE_ATM = 1.0
PRODUCTION_DURATION_NS = 0.01
PRODUCTION_TIMESTEP_FS = 2.0
PRODUCTION_REPORT_FRAMES = 100
FRICTION_PER_PS = 1.0
BAROSTAT_FREQUENCY_STEPS = 25
MINIMIZE_BEFORE_PRODUCTION = False
OPENMM_MINIMIZATION_MAX_ITERATIONS = 1_000

OPENMM_PLATFORM_NAME = "CUDA"
OPENMM_PLATFORM_PROPERTIES = {"Precision": "mixed", "DeviceIndex": "0"}
RANDOM_SEED = 51

SYSTEM_ROOT = OUTPUT_ROOT / SYSTEM_NAME
SDF_PATH = SYSTEM_ROOT / "sdf" / f"{SYSTEM_NAME}.sdf"
OPENMM_ROOT = SYSTEM_ROOT / "OpenMM"
PRODUCTION_START_DIR = OPENMM_ROOT / "production_start"
PRODUCTION_DIR = OPENMM_ROOT / "production_md"
PRODUCTION_DIR.mkdir(parents=True, exist_ok=True)

print(f"System: {SYSTEM_NAME}")
print(f"Input SDF: {SDF_PATH}")
print(f"Production directory: {PRODUCTION_DIR}")

## 2. Imports and Small Utilities

In [ ]:
from openff.interchange import Interchange
from openff.toolkit import ForceField, Molecule, Topology
from openff.toolkit.utils import ToolkitRegistry
from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper
from openff.units import unit as off_unit

import openmm
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
from openmm import unit as omm_unit
from openmm.app import DCDReporter, PDBxFile, StateDataReporter

DA_PER_NM3_TO_G_CM3 = 1.0 / 602.214076


def simulation_steps(duration_ns: float, timestep_fs: float) -> int:
    if duration_ns < 0:
        raise ValueError("duration_ns must be non-negative")
    if timestep_fs <= 0:
        raise ValueError("timestep_fs must be positive")
    return int(round(duration_ns * 1_000_000.0 / timestep_fs))


def report_interval(total_steps: int, n_frames: int) -> int:
    if n_frames <= 0:
        raise ValueError("n_frames must be positive")
    return max(1, total_steps // n_frames) if total_steps else 1


def openmm_platform_kwargs() -> dict:
    try:
        platform = openmm.Platform.getPlatformByName(OPENMM_PLATFORM_NAME)
    except Exception as exc:
        print(f"OpenMM {OPENMM_PLATFORM_NAME} platform unavailable; using default platform ({exc})")
        return {}
    return {"platform": platform, "platformProperties": OPENMM_PLATFORM_PROPERTIES}


def density_from_box_g_cm3(mass_da: float, box_vectors_nm: np.ndarray) -> float:
    volume_nm3 = abs(float(np.linalg.det(box_vectors_nm)))
    return mass_da * DA_PER_NM3_TO_G_CM3 / volume_nm3


def box_lengths_nm(box_vectors_nm: np.ndarray) -> np.ndarray:
    return np.array([np.linalg.norm(vector) for vector in box_vectors_nm], dtype=float)


def openmm_system_mass_da(system) -> float:
    return sum(system.getParticleMass(i).value_in_unit(omm_unit.dalton) for i in range(system.getNumParticles()))


def load_rdkit_sdf_records(path: Path) -> list[Chem.Mol]:
    if not path.exists():
        raise FileNotFoundError(f"Missing SDF cache: {path}")
    supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
    records = []
    for record_idx, mol in enumerate(supplier):
        if mol is None:
            raise ValueError(f"Could not read record {record_idx} from {path}")
        sanitized = Chem.Mol(mol)
        Chem.SanitizeMol(sanitized)
        records.append(sanitized)
    return records


def is_sodium_molecule(off_mol: Molecule) -> bool:
    return off_mol.n_atoms == 1 and off_mol.atom(0).symbol == "Na"

## 3. Load Cached SDF and Reconstruct MuPT

This is the first half of the round-trip: the SDF from the builder notebook reconstructs a MuPT `UNIVERSE` without rerunning DPD placement or chemistry construction.

In [ ]:
rdkit_mols = load_rdkit_sdf_records(SDF_PATH)
mupt_universe = primitive_from_mupt_sdf(SDF_PATH, reconstruct_bonds=False, reconstruct_shapes=True, label=f"{SYSTEM_NAME}_from_sdf")

segments = [node for node in PreOrderIter(mupt_universe) if node.role == PrimitiveRole.SEGMENT]
residues = [node for node in PreOrderIter(mupt_universe) if node.role == PrimitiveRole.RESIDUE]
particles = [node for node in PreOrderIter(mupt_universe) if node.role == PrimitiveRole.PARTICLE]
print(f"Loaded {len(rdkit_mols)} SDF records")
print(f"Reconstructed MuPT universe: segments={len(segments)}, residues={len(residues)}, particles={len(particles)}")

## 4. Build OpenFF/OpenMM System from Cached Molecules

The polymer melt has many copies of the same chain chemistry plus many sodium ions. To avoid repeated GNN charge assignment, this builds the topology from repeated representatives and applies charges from one polymer-chain template and one sodium template.

In [ ]:
def transfer_rdkit_metadata_to_openff(rdkit_mol: Chem.Mol, off_mol: Molecule) -> None:
    for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
        props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
        off_atom.metadata.update({
            "residue_name": str(props.get("residue_name", "UNK")),
            "residue_number": str(props.get("residue_id", props.get("mupt_residue_index", "1"))),
            "chain_id": str(props.get("chain_id", "A")),
            "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
        })


def representative_charge_molecules(off_molecules: list[Molecule], nagl_registry: ToolkitRegistry) -> list[Molecule]:
    chain_mol = next((mol for mol in off_molecules if not is_sodium_molecule(mol)), None)
    sodium_mol = next((mol for mol in off_molecules if is_sodium_molecule(mol)), None)
    if chain_mol is None:
        raise ValueError("No polymer chain molecule found for charge assignment")
    print(f"Assigning NAGL charges for representative polymer chain ({chain_mol.n_atoms} atoms)")
    chain_mol.assign_partial_charges(partial_charge_method=PARTIAL_CHARGE_METHOD, toolkit_registry=nagl_registry)
    charge_molecules = [chain_mol]
    if sodium_mol is not None:
        sodium_mol.partial_charges = [1.0] * off_unit.elementary_charge
        charge_molecules.append(sodium_mol)
    return charge_molecules


def representative_topology_and_positions(off_molecules: list[Molecule]):
    chain_template = next((mol for mol in off_molecules if not is_sodium_molecule(mol)), None)
    sodium_template = next((mol for mol in off_molecules if is_sodium_molecule(mol)), None)
    if chain_template is None:
        raise ValueError("No polymer chain molecule found for topology construction")

    topology_molecules = []
    positions = []
    for mol in off_molecules:
        template = sodium_template if is_sodium_molecule(mol) else chain_template
        if template is None or mol.n_atoms != template.n_atoms:
            raise ValueError(f"Molecule atom count {mol.n_atoms} does not match its representative template")
        topology_molecules.append(template)
        positions.append(mol.conformers[0].m_as(off_unit.angstrom))
    return Topology.from_molecules(topology_molecules), np.vstack(positions) * off_unit.angstrom


openff_molecules = []
for rdkit_mol in rdkit_mols:
    off_mol = Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True, hydrogens_are_explicit=True)
    transfer_rdkit_metadata_to_openff(rdkit_mol, off_mol)
    openff_molecules.append(off_mol)

ff = ForceField(FORCE_FIELD)
nagl_registry = ToolkitRegistry([NAGLToolkitWrapper()])
topology, topology_positions = representative_topology_and_positions(openff_molecules)
charge_molecules = representative_charge_molecules(openff_molecules, nagl_registry)

build_start_s = time.perf_counter()
interchange = ff.create_interchange(topology, charge_from_molecules=charge_molecules)
interchange.positions = topology_positions
print(f"Created Interchange with {interchange.topology.n_atoms} atoms in {time.perf_counter() - build_start_s:.2f} s")

## 5. Load Dense Production-Start Coordinates

The production-start `.npz` is deliberately used instead of a checkpoint. Rebuilding the OpenMM `System` from the SDF makes this notebook portable across machines, OpenMM platforms, and force object serialization details.

In [ ]:
def latest_manifest(path: Path) -> Path:
    manifests = sorted(path.glob("*_production_start_manifest.json"), key=lambda p: p.stat().st_mtime)
    if not manifests:
        raise FileNotFoundError(
            f"No production-start manifest found in {path}. Rerun the builder notebook with RUN_PERIODIC_NPT = True."
        )
    return manifests[-1]


manifest_path = latest_manifest(PRODUCTION_START_DIR)
manifest = json.loads(manifest_path.read_text())
arrays_path = EXAMPLES_ROOT / manifest["files"]["arrays_npz"]
arrays = np.load(arrays_path)
positions_nm = arrays["positions_nm"]
box_vectors_nm = arrays["box_vectors_nm"]

if positions_nm.shape[0] != interchange.topology.n_atoms:
    raise ValueError(f"Position count {positions_nm.shape[0]} does not match topology atom count {interchange.topology.n_atoms}")

interchange.positions = positions_nm * off_unit.nanometer
interchange.box = box_vectors_nm * off_unit.nanometer
print(f"Loaded production-start manifest: {manifest_path.relative_to(EXAMPLES_ROOT)}")
print(f"Loaded dense coordinates: {arrays_path.relative_to(EXAMPLES_ROOT)}")
print(f"Box lengths: {box_lengths_nm(box_vectors_nm)} nm")
print(f"Builder final density: {manifest.get('final_density_g_cm3', 'unknown')} g/cm^3")

## 6. Run Production OpenMM MD

In [ ]:
steps = simulation_steps(PRODUCTION_DURATION_NS, PRODUCTION_TIMESTEP_FS)
interval = report_interval(steps, PRODUCTION_REPORT_FRAMES)
temperature = PRODUCTION_TEMPERATURE_K * omm_unit.kelvin
pressure = PRODUCTION_PRESSURE_ATM * omm_unit.atmosphere
timestep = PRODUCTION_TIMESTEP_FS * omm_unit.femtosecond
friction = FRICTION_PER_PS / omm_unit.picosecond

integrator = LangevinMiddleIntegrator(temperature, friction, timestep)
integrator.setRandomNumberSeed(RANDOM_SEED)
barostat = MonteCarloBarostat(pressure, temperature, BAROSTAT_FREQUENCY_STEPS)
barostat.setRandomNumberSeed(RANDOM_SEED + 1)
simulation = interchange.to_openmm_simulation(
    integrator=integrator,
    combine_nonbonded_forces=False,
    additional_forces=[barostat],
    **openmm_platform_kwargs(),
)
print(f"OpenMM platform: {simulation.context.getPlatform().getName()}")

if MINIMIZE_BEFORE_PRODUCTION:
    print(f"Minimizing for up to {OPENMM_MINIMIZATION_MAX_ITERATIONS} iterations")
    simulation.minimizeEnergy(maxIterations=OPENMM_MINIMIZATION_MAX_ITERATIONS)

trajectory_path = PRODUCTION_DIR / f"ionomer_{SYSTEM_NAME}_production.dcd"
state_data_path = PRODUCTION_DIR / f"ionomer_{SYSTEM_NAME}_production_state_data.csv"
simulation.reporters.append(DCDReporter(str(trajectory_path), interval))
simulation.reporters.append(
    StateDataReporter(
        str(state_data_path),
        reportInterval=interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        temperature=True,
        volume=True,
        density=True,
        speed=True,
    )
)

print(f"Running {steps} steps ({PRODUCTION_DURATION_NS} ns), reporting every {interval} steps")
run_start_s = time.perf_counter()
if steps > 0:
    simulation.step(steps)
print(f"Production run completed in {time.perf_counter() - run_start_s:.2f} s")
print(f"Trajectory: {trajectory_path.relative_to(EXAMPLES_ROOT)}")
print(f"State data: {state_data_path.relative_to(EXAMPLES_ROOT)}")

## 7. Write Restart Files and Visualization Topology

In [ ]:
def metadata_openmm_topology(off_molecules: list[Molecule], box_vectors_nm: np.ndarray | None = None) -> openmm.app.Topology:
    topology = openmm.app.Topology()
    chains = {}
    residues = {}
    atom_lookup = {}
    for mol_idx, off_mol in enumerate(off_molecules):
        for atom_idx, off_atom in enumerate(off_mol.atoms):
            chain_id = str(off_atom.metadata.get("chain_id", str(mol_idx + 1)))
            residue_number = str(off_atom.metadata.get("residue_number", "1"))
            residue_name = str(off_atom.metadata.get("residue_name", "UNK"))
            atom_name = str(off_atom.metadata.get("atom_name", f"{off_atom.symbol}{atom_idx + 1}"))
            chain = chains.get(chain_id)
            if chain is None:
                chain = topology.addChain(chain_id)
                chains[chain_id] = chain
            residue_key = (chain_id, residue_number, residue_name)
            residue = residues.get(residue_key)
            if residue is None:
                residue = topology.addResidue(residue_name, chain, id=residue_number)
                residues[residue_key] = residue
            atom_lookup[(mol_idx, atom_idx)] = topology.addAtom(atom_name, openmm.app.element.get_by_symbol(off_atom.symbol), residue)
        for bond in off_mol.bonds:
            topology.addBond(atom_lookup[(mol_idx, bond.atom1_index)], atom_lookup[(mol_idx, bond.atom2_index)])
    if box_vectors_nm is not None:
        topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)
    return topology


state = simulation.context.getState(getEnergy=True, getPositions=True, getVelocities=True, enforcePeriodicBox=True)
final_positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
final_box_vectors_nm = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
system_mass_da = openmm_system_mass_da(simulation.system)
final_density = density_from_box_g_cm3(system_mass_da, final_box_vectors_nm)

final_state_path = PRODUCTION_DIR / f"ionomer_{SYSTEM_NAME}_production_final_state.xml"
final_checkpoint_path = PRODUCTION_DIR / f"ionomer_{SYSTEM_NAME}_production_final.chk"
final_arrays_path = PRODUCTION_DIR / f"ionomer_{SYSTEM_NAME}_production_final_arrays.npz"
final_topology_path = PRODUCTION_DIR / f"ionomer_{SYSTEM_NAME}_production_final_topology.cif"
final_manifest_path = PRODUCTION_DIR / f"ionomer_{SYSTEM_NAME}_production_manifest.json"

final_state_path.write_text(XmlSerializer.serialize(state))
simulation.saveCheckpoint(str(final_checkpoint_path))
np.savez_compressed(final_arrays_path, positions_nm=final_positions_nm, box_vectors_nm=final_box_vectors_nm)
topology_with_metadata = metadata_openmm_topology(openff_molecules, box_vectors_nm=final_box_vectors_nm)
with final_topology_path.open("w") as handle:
    PDBxFile.writeFile(topology_with_metadata, state.getPositions(asNumpy=True), handle)

production_manifest = {
    "system_name": SYSTEM_NAME,
    "source_sdf": str(SDF_PATH.relative_to(EXAMPLES_ROOT)),
    "source_production_start_manifest": str(manifest_path.relative_to(EXAMPLES_ROOT)),
    "duration_ns": float(PRODUCTION_DURATION_NS),
    "steps": int(steps),
    "temperature_k": float(PRODUCTION_TEMPERATURE_K),
    "pressure_atm": float(PRODUCTION_PRESSURE_ATM),
    "final_density_g_cm3": float(final_density),
    "box_lengths_nm": [float(x) for x in box_lengths_nm(final_box_vectors_nm)],
    "files": {
        "trajectory_dcd": str(trajectory_path.relative_to(EXAMPLES_ROOT)),
        "state_data_csv": str(state_data_path.relative_to(EXAMPLES_ROOT)),
        "final_state_xml": str(final_state_path.relative_to(EXAMPLES_ROOT)),
        "final_checkpoint": str(final_checkpoint_path.relative_to(EXAMPLES_ROOT)),
        "final_arrays_npz": str(final_arrays_path.relative_to(EXAMPLES_ROOT)),
        "final_topology_cif": str(final_topology_path.relative_to(EXAMPLES_ROOT)),
    },
}
final_manifest_path.write_text(json.dumps(production_manifest, indent=2) + "\n")

print(f"Final density: {final_density:.3f} g/cm^3")
print(f"Final topology: {final_topology_path.relative_to(EXAMPLES_ROOT)}")
print(f"Production manifest: {final_manifest_path.relative_to(EXAMPLES_ROOT)}")

## 8. Export Final SDF and Reconstruct MuPT Again

This is the second half of the round-trip: final OpenMM coordinates are written back into the MuPT SDF records, then MuPT reconstructs the hierarchy from that final SDF.

In [ ]:
def update_rdkit_records_from_positions(rdkit_records: list[Chem.Mol], positions_nm: np.ndarray) -> list[Chem.Mol]:
    updated = []
    cursor = 0
    for mol in rdkit_records:
        mol = Chem.Mol(mol)
        conf = mol.GetConformer()
        for atom_idx in range(mol.GetNumAtoms()):
            x, y, z = positions_nm[cursor] * 10.0
            conf.SetAtomPosition(atom_idx, Point3D(float(x), float(y), float(z)))
            cursor += 1
        updated.append(mol)
    if cursor != len(positions_nm):
        raise ValueError(f"Consumed {cursor} positions, but final array has {len(positions_nm)} atoms")
    return updated


final_sdf_path = PRODUCTION_DIR / f"{SYSTEM_NAME}_production_final.sdf"
final_rdkit_mols = update_rdkit_records_from_positions(rdkit_mols, final_positions_nm)
writer = Chem.SDWriter(str(final_sdf_path))
for mol in final_rdkit_mols:
    writer.write(mol)
writer.close()
production_manifest["files"]["final_sdf"] = str(final_sdf_path.relative_to(EXAMPLES_ROOT))
final_manifest_path.write_text(json.dumps(production_manifest, indent=2) + "\n")

final_mupt_universe = primitive_from_mupt_sdf(final_sdf_path, reconstruct_bonds=False, reconstruct_shapes=True, label=f"{SYSTEM_NAME}_production_final")
final_segments = [node for node in PreOrderIter(final_mupt_universe) if node.role == PrimitiveRole.SEGMENT]
final_particles = [node for node in PreOrderIter(final_mupt_universe) if node.role == PrimitiveRole.PARTICLE]
print(f"Final SDF: {final_sdf_path.relative_to(EXAMPLES_ROOT)}")
print(f"Reconstructed final MuPT universe: segments={len(final_segments)}, particles={len(final_particles)}")

## What This Demonstrates

The builder notebook demonstrates MuPT system construction, DPD-based placement, cached SDF/mmCIF export, and dense OpenMM initialization. This companion demonstrates that the cached SDF is sufficient to reconstruct the MuPT hierarchy and OpenFF molecules, while the `production_start` arrays provide portable periodic coordinates for production MD. The final SDF closes the loop by bringing the OpenMM production coordinates back into MuPT-readable form.